In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create some dummy data
x = np.linspace(0, 10, 100)
y = np.sin(x)

print("Libraries are working perfectly!")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x, y, label='Sine Wave')
plt.title("Environment Test: Plotting Check")
plt.xlabel("X-axis")
plt.ylabel("Y-axis")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import holoviews as hv
from holoviews import streams, opts

# Activate the Bokeh backend (required for interactivity)
hv.extension('bokeh')

# --- 1. Scientific Definitions (Simplified 1D Rectangular Barrier) ---
# Goal: Illustrate the effect of the Gamow Factor (Psi squared)
# V(x) = V_0 for 0 <= x <= a, else V(x) = 0.
# Particle energy E < V_0.

hbar_me_factor = 0.0381  # hbar^2 / (2 * electron mass) in units of (eV * nm^2)

def wave_function_components(E, V0, mass_factor=hbar_me_factor):
    """
    Computes the wave function components for a particle of energy E
    encountering a rectangular barrier V0 at x=0.
    """
    # Incident particle region (x < 0)
    # Re(psi) = cos(k1*x) + Im(psi)=sin(k1*x) (reflected wave simplified)
    k1 = np.sqrt(E / mass_factor) if E > 0 else 0
    
    # Barrier Region (0 <= x <= a). Simplified length a=1 nm.
    a_len = 1.0 # nm
    
    # If energy is below barrier, k2 is imaginary, causing exponential decay
    kappa = np.sqrt((V0 - E) / mass_factor) if V0 > E else 0
    decay_amplitude = np.exp(-kappa * a_len) # Decay factor across the barrier
    
    # Generate data points
    x = np.linspace(-3, 3, 400) # nm
    y_psi = np.zeros_like(x)
    y_prob = np.zeros_like(x)
    y_V = np.zeros_like(x)

    # Calculate values
    region1 = (x < 0)
    region2 = (x >= 0) & (x < a_len)
    region3 = (x >= a_len)

    # Simple model focused on showing the decay
    y_psi[region1] = np.sin(k1 * x[region1])
    y_prob[region1] = np.abs(np.sin(k1 * x[region1]))**2
    
    # Barrier region decay
    decay_curve = np.exp(-kappa * x[region2])
    y_psi[region2] = decay_curve
    y_prob[region2] = np.abs(decay_curve)**2
    y_V[region2] = V0

    # Transmitted region
    transmitted_curve = decay_amplitude * np.sin(k1 * (x[region3] - a_len))
    y_psi[region3] = transmitted_curve
    y_prob[region3] = np.abs(transmitted_curve)**2

    return x, y_psi, y_prob, y_V, decay_amplitude**2

# --- 2. Create Interactivity using HoloViews DynamicMap ---

# Parameters to adjust
energies = np.linspace(0.1, 4.9, 50)
barriers = np.linspace(3.0, 10.0, 50)

def tunneling_viz(Energy, Barrier):
    """
    Dynamic function that returns updated plots based on slider input.
    """
    x, psi, prob, V, T = wave_function_components(Energy, Barrier)
    
    # Calculate Gamow type exponential factor
    gamow_factor = np.exp(-2 * np.sqrt((Barrier - Energy) / hbar_me_factor) * 1.0) # 1nm length
    
    # 1. Wave Function Plot (Blue)
    p_psi = hv.Curve((x, psi), 'x (nm)', 'ψ(x)').opts(
        title=f"Wave Function (Re): ψ(x) Decay across Barrier. T ~ {T:.3f}",
        line_width=2, color='blue', alpha=0.5
    )
    
    # 2. Probability Density Plot (Green)
    p_prob = hv.Area((x, prob), 'x (nm)', '|ψ(x)|²').opts(
        title="Probability Density: Note non-zero probability at x > 1 nm",
        color='green', alpha=0.3
    )

    # 3. Barrier Visualization (Gray area)
    p_V = hv.Area((x, V)).opts(
        color='gray', alpha=0.15, xlabel='x (nm)', ylabel='Energy (eV)'
    )
    
    # 4. Reference line for Particle Energy (Red)
    p_E = hv.HLine(Energy).opts(color='red', line_dash='dashed', alpha=0.8)

    # 5. Label for the Gamow Factor
    gamow_label = hv.Text(2.2, Barrier + 0.5, f"Gamow Barrier Factor:\n ~ {gamow_label:.3e}").opts(text_font_size='9pt')
    
    return (p_V * p_E * p_prob * p_psi).opts(width=800, height=500, ylim=(0, 10.5))

# --- 3. Run and Visualize ---

# Create the dynamic map which generates the interactive widgets
dmap = hv.DynamicMap(tunneling_viz, kdims=['Energy', 'Barrier'])

# Define slider ranges and steps
dmap.redim.range(Energy=(0.1, 9.9), Barrier=(3.0, 10.0))
dmap.redim.step(Energy=0.1, Barrier=0.1)
dmap.redim.default(Energy=2.5, Barrier=5.0)

dmap